In [ ]:
import sys
sys.path.append('../src')

from llm import LLM

/home/jlllsarcinelli/exp/pseudo-task-agents-dev/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-02 10:28:56,902	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
from utils import load_yaml
from datasets import load_dataset


dataset_name = 'ohsumed-single'
dataset_config = load_yaml(f'../configs/datasets/{dataset_name}.yaml')

dataset = load_dataset(dataset_config['path'])
texts = list(dataset['test']['text'])
labels = list(dataset['test']['label'])
id2label = dataset_config['id2label']
class_task_desc = dataset_config['labeling_task_description']


In [3]:
from topology import build_pyg_data
import torch


data = build_pyg_data(texts, k=3, model_name='Qwen/Qwen3-VL-Embedding-2B')
data.y = torch.tensor(labels)
data.id2label = id2label

Loading weights: 100%|██████████| 625/625 [00:00<00:00, 22726.75it/s]
Default prompt name is set to 'default'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


In [4]:
# model_path = 'google/gemma-4-E4B-it'
# llm = LLM(model_path, load_model=True, gpu_memory_utilization=0.8)

In [5]:
import re


def get_questions(class_task_desc, llm, prev_questions=[], suggestions=None):
    prev_questions_str = (
        "\n\nYou already have found these useful questions:\n\n"
        '\n'.join(prev_questions) + "\n\n"
        "Find other questions similar to these but that are diverse and activate on different texts. Do NOT repeat them\n\n"
    ) if prev_questions else ''

    suggestions_str = (
        "A manager agent analyzed the previously generated questions and wrote the following report:\n\n"
        "```\n"
        f"{suggestions}\n"
        "```\n\n"
        "Incorporate these suggestions from now on.\n\n"
    ) if suggestions else ''

    prompt = (
        f"You are a pseudo-task generator that generates support tasks for a main classification task "
        f"in an iterative process. Here's the main classification task:\n\n"
        f"Task:\n\n"
        f"```\n"
        f"{class_task_desc}\n"
        f"```\n\n"
        f"{prev_questions_str}"
        f"Now, based on the main task, generate new questions "
        f"that will progressively narrow down the class and help guiding the classification process."
        f"Questions must be binary or categorical, be designed to discriminate class domains and avoid trivial triggers.\n"
        f"Return a reasoning and a plan, followed by the questions in the following format:\n\n"
        f"```\n"
        f"Reasoning:\n\n"
        f"<your reasoning>:\n\n"
        f"Planing:\n\n"
        f"<your plan>\n\n"
        f"Questions:\n\n"
        f"1) <question_1>\n\n"
        f"2) <question_2>\n\n"
        f"3) <question_3>\n\n"
        f"...\n\n"
        f"n) <question_n>\n\n"
        f"```\n\n"
        f"NOTE: generate up to 5 questions. Provide ONLY the questions. Questions must not be hierarchical."
    )
    
    response = llm.invoke(prompt)

    def parse_response(response):
        pattern = r"^\d+\)\s+(.*?)(?=\n\d+\)|$)"
        
        questions = re.findall(pattern, response, flags=re.MULTILINE)
        
        return [q.strip() for q in questions]

    questions = parse_response(response)

    return response, questions


def get_questions_for_class(class_task_desc, class_labels, llm):

    prompts = []
    for c in class_labels:
        prompt = (
            f"You are a pseudo-task generator that generates support tasks for a main classification task "
            f"in an iterative process. Here's the main classification task:\n\n"
            f"Task:\n\n"
            f"```\n"
            f"{class_task_desc}\n"
            f"```\n\n"
            f"Now, based on the main task, generate 3 questions "
            f"that will progressively narrow down the class of a text and help guiding the classification process for the \"{c}\" class.\n"
            f"Questions must be binary or categorical, be designed to discriminate class domains and avoid trivial triggers.\n"
            f"Return a reasoning and a plan, followed by the questions in the following format:\n\n"
            f"```\n"
            f"Reasoning:\n\n"
            f"<your reasoning>:\n\n"
            f"Planing:\n\n"
            f"<your plan>\n\n"
            f"Questions:\n\n"
            f"1) <question_1>\n\n"
            f"2) <question_2>\n\n"
            f"3) <question_3>\n\n"
        )

        prompts.append(prompt)
    
    responses = llm.invoke(prompts)

    def parse_response(response):
        pattern = r"^\d+\)\s+(.*?)(?=\n\d+\)|$)"
        
        questions = re.findall(pattern, response, flags=re.MULTILINE)
        
        return [q.strip() for q in questions]

    questions = []
    for response in responses:
        questions += parse_response(response)

    return responses, questions


def get_answers(class_task_desc, questions, texts, sampled_indices, llm):
    questions_str = '\n'.join(f'{i+1}) {question}' for i,question in enumerate(questions))

    template = (
        f"Task:\n\n"
        f"```\n"
        f"{class_task_desc}\n"
        f"```\n\n"
        f'Text: "{{text}}"\n\n'
        f"Based on the task, answer these questions with ONLY yes or no and classify this text using the respective label indices:\n\n"
        f"{questions_str}\n\n"
        f"Respond STRICTLY in the following JSON format:\n\n"
        f"{{{{\n"
        f'    "q1": "<yes or no>",\n'
        f'    "q2": "<yes or no>",\n'
        f'    "q3": "<yes or no>",\n'
        f'    ...'
        f'    "qn": "<yes or no>",\n'
        f'    "label_id": "label index integer"\n'
        f"}}}}"
    )

    prompts = [template.format(text=texts[idx]) for idx in sampled_indices]
    responses = llm.invoke(prompts, temperature=0.0)

    def parse(response):
        answers_pat = r'["q[0-9]+": "([^"]+)"[^"]*]*'
        label_pat = r'"label_id": ["]*([^"}\s]+)["]*'

        try:
            answers = [a.strip().lower().startswith('y') for a in re.findall(answers_pat, response)]
            
            if len(answers) != len(questions):
                answers = [False for _ in questions]

            label_id = re.search(label_pat, response).group(1).strip()
            label = int(label_id) if label_id else 0
        except Exception as e:
            answers = [False for _ in questions]
            label = 0
        
        return {
            "answers": answers,
            "label": label
        }

    answers = { idx: parse(response.strip()) for idx,response in zip(sampled_indices, responses) }

    return answers

In [6]:
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy as scipy_entropy


def smoothed_entropy_binary(q, alpha=1.0):
    # print('smoothed_entropy_binary')
    """
    q: binary array shape (n,)
    alpha: Laplace smoothing parameter
    """
    n = len(q)
    k = np.sum(q)

    p = 0 if n == 0 else (k + alpha) / (n + 2 * alpha)
    probs = np.array([p, 1 - p])
    return scipy_entropy(probs, base=2)


def cohesion_separation_score(q, embeddings):
    # print('cohesion_separation_score')
    """
    q: binary array shape (n,)
    embeddings: (n, d)
    """
    idx1 = np.where(q == 1)[0]
    idx0 = np.where(q == 0)[0]

    if len(idx1) < 2 or len(idx0) < 2:
        return 0.0

    emb1 = embeddings[idx1]
    emb0 = embeddings[idx0]

    sim11 = cosine_similarity(emb1)
    sim00 = cosine_similarity(emb0)
    sim10 = cosine_similarity(emb1, emb0)

    intra1 = (np.sum(sim11) - len(idx1)) / (len(idx1)*(len(idx1)-1))
    intra0 = (np.sum(sim00) - len(idx0)) / (len(idx0)*(len(idx0)-1))
    inter = np.mean(sim10)

    return ((intra1 + intra0) / 2.0) - inter


import torch
from torch_geometric.utils import k_hop_subgraph, to_undirected

def graph_cut_ratio(
    answers: torch.Tensor,
    edge_index: torch.Tensor,
    sample_indices: torch.Tensor,
    k_hop: int = 1
):
    """
    answers: (N,) tensor with binary values for sample nodes
             and -1 for unlabeled nodes
    edge_index: (2, E)
    sample_indices: indices that have valid answers
    k_hop: neighborhood size

    Returns:
        cut_ratio over induced neighborhood subgraph
    """

    edge_index = to_undirected(edge_index)
    num_nodes = answers.size(0)

    # Expand all sample nodes at once
    subset, sub_edge_index, _, _ = k_hop_subgraph(
        sample_indices,
        k_hop,
        edge_index,
        relabel_nodes=False
    )

    if sub_edge_index.size(1) == 0:
        return 0.0

    src, dst = sub_edge_index

    # Only consider edges where BOTH nodes have valid answers
    valid_mask = (answers[src] != -1) & (answers[dst] != -1)

    if valid_mask.sum() == 0:
        return 0.0

    src = src[valid_mask]
    dst = dst[valid_mask]

    total_edges = src.size(0)
    cut_edges = (answers[src] != answers[dst]).sum().item()

    return cut_edges / total_edges


def composite_informativeness(
    q_index,
    answers,
    embeddings_sample,
    edge_index,
    sample_indices,
    num_nodes,
    alpha=1.0,
    w_entropy=1.0,
    w_cohesion=1.0,
    w_cut=1.0
):
    q = answers[:, q_index]

    H = smoothed_entropy_binary(q, alpha)
    cohesion = cohesion_separation_score(q, embeddings_sample)

    q_y = torch.full((num_nodes,), -1)
    q_y[sample_indices] = torch.tensor(q, dtype=torch.long)
    graph_cut = graph_cut_ratio(q_y, edge_index, sample_indices)

    score = (
        w_entropy * H +
        w_cohesion * cohesion -
        w_cut * graph_cut
    )

    return {
        "entropy": np.array(H).item(),
        "cohesion_gap": np.array(cohesion).item(),
        "graph_cut": np.array(graph_cut).item(),
        "composite_score": np.array(score).item()
    }


def evaluate_all_questions(
    answers,
    embeddings_sample,
    edge_index,
    sample_indices,
    num_nodes,
    alpha=1.0,
    weights=None
):
    if weights is None:
        weights = {
            "entropy": 1.0,
            "cohesion": 1.0,
            "graph_cut": 1.0,
        }

    results = []

    for q_index in range(answers.shape[1]):
        metrics = composite_informativeness(
            q_index,
            answers,
            embeddings_sample,
            edge_index,
            sample_indices,
            num_nodes,
            alpha=alpha,
            w_entropy=weights["entropy"],
            w_cohesion=weights["cohesion"],
            w_cut=weights["graph_cut"]
        )
        metrics["question_index"] = q_index
        results.append(metrics)

    return sorted(results, key=lambda x: x["composite_score"], reverse=True)

In [7]:
import matplotlib.pyplot as plt

import numpy as np
import matplotlib.pyplot as plt

def plot_all_metrics(results, top_k=None):
    """
    results: output from evaluate_all_questions()
    top_k: optionally restrict to top_k questions
    """

    if top_k is not None:
        results = results[:top_k]

    question_indices = [r["question_index"] for r in results]

    entropy = [r["entropy"] for r in results]
    cohesion = [r["cohesion_gap"] for r in results]
    graph_cut = [r["graph_cut"] for r in results]
    composite = [r["composite_score"] for r in results]

    x = np.arange(len(results))
    width = 0.13

    plt.figure(figsize=(14, 6))

    plt.bar(x - 2.5*width, entropy, width, label="Entropy")
    plt.bar(x - 1.5*width, cohesion, width, label="Cohesion Gap")
    plt.bar(x - 0.5*width, graph_cut, width, label="Graph Cut Ratio")
    plt.bar(x + 0.5*width, composite, width, label="Composite")

    plt.xticks(x, question_indices)
    plt.xlabel("Question Index")
    plt.ylabel("Metric Value")
    plt.title("All Informativeness Metrics per Question")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [8]:
import torch

def pairwise_jaccard(answers):
    """
    answers: (Q, N) binary tensor
    returns: (Q, Q) similarity matrix
    """
    Q = answers.size(0)

    intersection = torch.matmul(answers.float(), answers.float().t())
    row_sums = answers.sum(dim=1).unsqueeze(1)
    union = row_sums + row_sums.t() - intersection

    sim = intersection / (union + 1e-8)

    return sim


def select_dissimilar_questions(
    answers,
    similarity_threshold=0.85,
    order="entropy"  # or "activation"
):
    """
    answers: (Q, N) binary tensor
    similarity_threshold: max allowed Jaccard similarity
    order:
        - "entropy": prioritize high entropy questions
        - "activation": prioritize balanced activation size
        - None: original order

    Returns:
        selected_indices (list)
    """

    Q = answers.size(0)
    sim = pairwise_jaccard(answers)

    # Determine ordering
    if order == "entropy":
        p = answers.float().mean(dim=1)
        entropy = -(p * torch.log(p + 1e-8) + (1 - p) * torch.log(1 - p + 1e-8))
        order_idx = torch.argsort(entropy, descending=True)

    elif order == "activation":
        activation_size = answers.sum(dim=1)
        order_idx = torch.argsort(
            torch.abs(activation_size - answers.size(1)/2)
        )

    else:
        order_idx = torch.arange(Q)

    selected = []

    for idx in order_idx.tolist():

        keep = True
        for s in selected:
            if sim[idx, s] >= similarity_threshold:
                keep = False
                break

        if keep:
            selected.append(idx)

    return selected

In [9]:
import torch

def compute_mutual_information(binary_q, labels, num_classes):
    """
    binary_q: (N,) {0,1}
    labels: (N,) int class labels
    """

    N = labels.size(0)
    mi = 0.0

    for q_val in [0, 1]:
        mask_q = (binary_q == q_val)
        p_q = mask_q.float().mean()

        if p_q == 0:
            continue

        for c in range(num_classes):
            mask_joint = mask_q & (labels == c)
            p_joint = mask_joint.float().mean()

            if p_joint == 0:
                continue

            p_c = (labels == c).float().mean()

            mi += p_joint * torch.log(p_joint / (p_q * p_c) + 1e-8)

    return mi.item()


def evaluate_question_relevance(
    answers,
    labels,
    mi_threshold=0.001,
    min_class_activation_diff=0.05
):
    """
    answers: (Q, N) binary
    labels: (N,) int
    Returns:
        relevance_info: list of dicts
        keep_indices: list
        remove_indices: list
    """

    Q, N = answers.shape
    num_classes = labels.unique().numel()

    relevance_info = []
    keep_indices = []
    remove_indices = []

    for q in range(Q):
        binary_q = answers[q]

        # 1️⃣ Mutual Information
        mi = compute_mutual_information(binary_q, labels, num_classes)

        # 2️⃣ Per-class activation rates
        class_rates = []
        for c in range(num_classes):
            mask_c = (labels == c)
            if mask_c.sum() == 0:
                class_rates.append(0.0)
            else:
                rate = binary_q[mask_c].float().mean().item()
                class_rates.append(rate)

        max_diff = max(class_rates) - min(class_rates)

        # Decision rule
        keep = (mi >= mi_threshold) and (max_diff >= min_class_activation_diff)

        if keep:
            keep_indices.append(q)
        else:
            remove_indices.append(q)

        relevance_info.append({
            "question_index": q,
            "mutual_information": mi,
            "max_class_activation_diff": max_diff,
            "class_rates": class_rates,
            "keep": keep
        })

    return relevance_info, keep_indices, remove_indices

In [10]:
import torch
import torch.nn.functional as F
from torch_geometric.utils import to_undirected

def per_class_similar_conflicts(
    data,
    labels,
    similarity_threshold=0.8,
    n_per_class=5
):
    """
    data: PyG Data object (must contain x and edge_index)
    labels: (N,) predicted labels
    similarity_threshold: cosine similarity threshold
    n_per_class: max examples per class

    Returns:
        dict[class_id] = list of (u, v, sim)
    """

    edge_index = to_undirected(data.edge_index)
    x = data.x
    src, dst = edge_index

    # Normalize embeddings
    x_norm = F.normalize(x, p=2, dim=1)

    # Cosine similarity for connected pairs
    sims = (x_norm[src] * x_norm[dst]).sum(dim=1)

    num_classes = labels.unique().numel()
    results = {c: [] for c in labels.unique().tolist()}

    for i in range(src.size(0)):
        u = src[i].item()
        v = dst[i].item()
        sim = sims[i].item()

        if sim < similarity_threshold:
            continue

        if labels[u] == labels[v]:
            continue

        # Assign conflict to both involved classes
        for node, other in [(u, v), (v, u)]:
            c = labels[node].item()

            if len(results[c]) < n_per_class:
                results[c].append((node, other, sim))

    return results

In [11]:
import torch
import torch.nn.functional as F

def compute_instance_jaccard(answers):
    """
    answers: (Q, N)
    returns: (N, N) similarity matrix
    """
    A = answers.t().float()  # (N, Q)

    intersection = torch.matmul(A, A.t())
    row_sums = A.sum(dim=1).unsqueeze(1)
    union = row_sums + row_sums.t() - intersection

    sim = intersection / (union + 1e-8)
    return sim


def find_answer_similar_label_conflicts(
    answers,
    labels,
    similarity_threshold=0.8,
    n_per_class=5,
    similarity="jaccard"  # or "cosine"
):
    """
    answers: (Q, N) binary
    labels: (N,)
    similarity_threshold: similarity cutoff
    n_per_class: max returned per class
    similarity: "jaccard" or "cosine"

    Returns:
        dict[class_id] = list of (i, j, similarity)
    """

    N = labels.size(0)

    if similarity == "jaccard":
        sim_matrix = compute_instance_jaccard(answers)

    elif similarity == "cosine":
        A = F.normalize(answers.t().float(), dim=1)
        sim_matrix = torch.matmul(A, A.t())

    else:
        raise ValueError("similarity must be 'jaccard' or 'cosine'")

    results = {c: [] for c in labels.unique().tolist()}

    # Only check upper triangle to avoid duplicates
    for i in range(N):
        for j in range(i + 1, N):

            sim = sim_matrix[i, j].item()

            if sim < similarity_threshold:
                continue

            if labels[i] == labels[j]:
                continue

            # assign to both classes
            for node, other in [(i, j), (j, i)]:
                c = labels[node].item()
                if len(results[c]) < n_per_class:
                    results[c].append((node, other, sim))

    return results

In [12]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

def cluster_and_plot_answers_with_labels(
    answers,
    predicted_labels,
    n_clusters=5
):
    """
    answers: Tensor (Q, N) binary
    predicted_labels: Tensor (N,)
    n_clusters: number of clusters for KMeans (used only for structure)

    Returns:
        cluster_labels
    """

    # Convert (Q, N) -> (N, Q)
    X = answers.t().float().cpu().numpy()
    predicted_labels = predicted_labels.cpu().numpy()

    # ---- Cluster (structure only) ----
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X)

    # ---- Reduce to 2D ----
    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X)

    # ---- Plot colored by predicted labels ----
    plt.figure(figsize=(8, 6))

    plt.scatter(
        X_2d[:, 0],
        X_2d[:, 1],
        c=predicted_labels
    )

    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.title("Answer-Based Structure Colored by Predicted Labels")
    plt.tight_layout()
    plt.show()

    return cluster_labels

In [13]:
llm = LLM('AxionML/Qwen3.5-9B-NVFP4', load_model=True, gpu_memory_utilization=0.8)
chat_kwargs = {
    'chat_template_kwargs': {
        'enable_thinking': False
    }
}
llm.set_default_chat_kwargs(chat_kwargs)

INFO 05-02 10:30:08 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': 'AxionML/Qwen3.5-9B-NVFP4'}
INFO 05-02 10:30:09 [model.py:549] Resolved architecture: Qwen3_5ForConditionalGeneration
INFO 05-02 10:30:09 [model.py:1678] Using max model len 4096
INFO 05-02 10:30:11 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.


[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.


INFO 05-02 10:30:11 [config.py:281] Setting attention block size to 528 tokens to ensure that attention page size is >= mamba page size.
INFO 05-02 10:30:11 [config.py:312] Padding mamba page size by 0.76% to ensure that mamba page size and attention page size are exactly equal.
WARNING 05-02 10:30:11 [modelopt.py:998] Detected ModelOpt NVFP4 checkpoint. Please note that the format is experimental and could change in future.
INFO 05-02 10:30:11 [vllm.py:790] Asynchronous scheduling is enabled.
INFO 05-02 10:30:11 [compilation.py:292] Enabled custom fusions: act_quant


[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


WARNING 05-02 10:30:23 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=3985335) INFO 05-02 10:30:34 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='AxionML/Qwen3.5-9B-NVFP4', speculative_config=None, tokenizer='AxionML/Qwen3.5-9B-NVFP4', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=modelopt_fp4, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=Stru

...

(EngineCore pid=3985335) INFO 05-02 10:30:52 [default_loader.py:384] Loading weights took 5.35 seconds
(EngineCore pid=3985335) WARNING 05-02 10:30:52 [nvfp4_utils.py:144] Your GPU does not have native support for FP4 computation but FP4 quantization is being used. Weight-only FP4 compression will be used leveraging the Marlin kernel. This may degrade performance for compute-heavy workloads.
(EngineCore pid=3985335) INFO 05-02 10:30:53 [gpu_model_runner.py:4820] Model loading took 8.42 GiB memory and 6.628913 seconds
(EngineCore pid=3985335) INFO 05-02 10:30:53 [gpu_model_runner.py:5753] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 image items of the maximum feature size.
(EngineCore pid=3985335) INFO 05-02 10:30:57 [backends.py:1051] Using cache directory: /home/jlllsarcinelli/.cache/vllm/torch_compile_cache/184b33c8db/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=3985335) INFO 05-02 10:30:57 [backends.py:1111] Dynamo bytecode trans

(EngineCore pid=3985335) /home/jlllsarcinelli/exp/pseudo-task-agents-dev/.venv/lib/python3.12/site-packages/vllm/model_executor/layers/fla/ops/utils.py:113: UserWarning: Input tensor shape suggests potential format mismatch: seq_len (16) < num_heads (32). This may indicate the inputs were passed in head-first format [B, H, T, ...] when head_first=False was specified. Please verify your input tensor format matches the expected shape [B, T, H, ...].
(EngineCore pid=3985335)   return fn(*contiguous_args, **contiguous_kwargs)


(EngineCore pid=3985335) INFO 05-02 10:31:00 [monitor.py:76] Initial profiling/warmup run took 0.89 s
(EngineCore pid=3985335) INFO 05-02 10:31:00 [kv_cache_utils.py:829] Overriding num_gpu_blocks=0 with num_gpu_blocks_override=512
(EngineCore pid=3985335) INFO 05-02 10:31:00 [gpu_model_runner.py:5876] Profiling CUDA graph memory: PIECEWISE=51 (largest=512), FULL=35 (largest=256)
(EngineCore pid=3985335) INFO 05-02 10:31:01 [gpu_model_runner.py:5955] Estimated CUDA graph memory: 0.71 GiB total
(EngineCore pid=3985335) INFO 05-02 10:31:01 [gpu_worker.py:436] Available KV cache memory: 8.38 GiB
(EngineCore pid=3985335) INFO 05-02 10:31:01 [gpu_worker.py:470] In v0.19, CUDA graph memory profiling will be enabled by default (VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=1), which more accurately accounts for CUDA graph memory during KV cache allocation. To try it now, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=1 and increase --gpu-memory-utilization from 0.8000 to 0.8304 to maintain the same 

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 29.90it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 34.55it/s]


(EngineCore pid=3985335) INFO 05-02 10:31:04 [gpu_model_runner.py:6046] Graph capturing finished in 3 secs, took 0.60 GiB
(EngineCore pid=3985335) INFO 05-02 10:31:04 [gpu_worker.py:597] CUDA graph pool memory: 0.6 GiB (actual), 0.71 GiB (estimated), difference: 0.12 GiB (19.6%).
(EngineCore pid=3985335) INFO 05-02 10:31:04 [core.py:283] init engine (profile, create kv cache, warmup model) took 11.63 seconds
(EngineCore pid=3985335) INFO 05-02 10:31:05 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=3985335) INFO 05-02 10:31:05 [compilation.py:292] Enabled custom fusions: act_quant


In [28]:
import numpy as np
from sklearn.metrics import classification_report
import random
from utils import kmeans_sample
from topology import build_hetero_graph_binary, build_hetero_graph_binary_link_pred
from agents.gnn import (HeteroGNN, train_hetero, evaluate_hetero,
                        MultiAnswerHeteroGNN, 
                        MultiTaskGNN, train_multitask, evaluate_multitask,
                        HeteroLinkPredictor, train_link_predictor,
                        GNN, train_model, evaluate_model)


embeddings = data.x.to(torch.float).detach().cpu().numpy()
sample_indices = kmeans_sample(embeddings, 100)

true = np.array(labels)[sample_indices]

max_iter = 3

existing_questions = []
suggestions = None
for _ in range(max_iter):
    # response, questions = get_questions(class_task_desc, llm, existing_questions)
    response, questions = get_questions_for_class(class_task_desc, id2label.values(), llm)

    print('Using:')
    for q in questions:
        print('\t* ', q)

    for q in existing_questions:
        print('\t' + q)
    print()

    existing_questions += questions

    answers_dict = get_answers(class_task_desc, existing_questions, texts, sample_indices, llm)

    llm.unload_model()

    answers = np.array([a['answers'] for a in answers_dict.values()])
    answers_y = torch.full((len(texts), len(existing_questions)), -1)
    answers_y[sample_indices] = torch.tensor(answers, dtype=torch.long)

    pseudo_labels = np.array([a['label'] for a in answers_dict.values()])

    pseudo_y = torch.full((len(texts),), -1)
    for idx in answers_dict:
        l = answers_dict[idx]['label']
        pseudo_y[idx] = l if l in id2label.keys() else 0

    data.pseudo_y = pseudo_y

    # hetero_data = build_hetero_graph_binary_link_pred(data, existing_questions, answers_y, pseudo_y, torch.tensor(labels), model_name='Qwen/Qwen3-VL-Embedding-2B')
    # hetero_data = build_hetero_graph_binary(data, existing_questions, answers_y, pseudo_y, torch.tensor(labels), model_name='Qwen/Qwen3-VL-Embedding-2B')

    # model = GNN(data.x.shape[1], hidden_channels=256, out_channels=int(data.y.max() + 1))
    # train_model(model, data, 200)
    # report, _ = evaluate_model(model, data)

    data.answers = answers_y

    model = MultiTaskGNN(data.x.shape[1], hidden_dim=256, num_classes=int(data.y.max() + 1), num_questions=len(existing_questions))
    model = train_multitask(model, data)
    report, _, _, propagated_y = evaluate_multitask(model, data)

    # model = HeteroGNN(hidden_dim=256, out_dim=int(hetero_data['text'].y.max() + 1))
    # model = train_hetero(model, hetero_data, epochs=200, lr=1e-3)
    # report, _, propagated_y = evaluate_hetero(model, hetero_data)

    print(report)

    # conflict_edges = per_class_similar_conflicts(
    #     data,
    #     torch.tensor(propagated_y)
    # )

    # conflicts = find_answer_similar_label_conflicts(
    #     torch.tensor(answers_arr).T,
    #     torch.tensor(pseudo_labels),
    #     similarity_threshold=0.6,
    #     n_per_class=5,
    #     similarity="jaccard"
    # )

    # cluster_and_plot_answers_with_labels(torch.tensor(answers_arr).T, torch.tensor(pseudo_labels))

    # print(answers_arr.shape)

    # for c, pairs in conflicts.items():
    #     print(f"\nClass {c}:")
    #     for a, b, _ in pairs:
    #         label_a = id2label[propagated_y[a]]
    #         label_b = id2label[propagated_y[b]]
    #         true_a = id2label[labels[a]]
    #         true_b = id2label[labels[b]]
    #         text_a = '\n\t'.join(texts[a].split('\n'))
    #         text_b = '\n\t'.join(texts[b].split('\n'))

    # for c, examples in conflict_edges.items():
    #     print(f'Class = {id2label[c]}')
    #     for a,b,_ in examples:
    #         label_a = id2label[propagated_y[a]]
    #         label_b = id2label[propagated_y[b]]
    #         true_a = id2label[labels[a]]
    #         true_b = id2label[labels[b]]
    #         text_a = '\n\t'.join(texts[a].split('\n'))
    #         text_b = '\n\t'.join(texts[b].split('\n'))

    #     print(f'Conflict:\n```\n\t{text_a}\n```\n(predicted {label_a}, true {true_a})\n```\n\t{text_b}\n```\n(precited {label_b}, true {true_b})\n')


    # report = classification_report(true, pred, target_names=list(id2label.values())[:true.max()], zero_division=0)
    # print(report)

    # questions_evalutaion = evaluate_all_questions(answers, data.x.detach().cpu()[sample_indices], data.edge_index.detach().cpu(), sample_indices, len(texts))

    # plot_all_metrics(questions_evalutaion)

    # dissimilar_questions = select_dissimilar_questions(torch.tensor(answers_arr).T)

    # questions_evalutaion = sorted(questions_evalutaion, key=lambda q: q['cohesion_gap'])
    # good_cohesion_question_indices = [e['question_index'] for e in questions_evalutaion[:5]]
    # good_cohesion_questions = [q for i,q in enumerate(existing_questions) if i in good_cohesion_question_indices]

    # existing_questions = good_cohesion_questions
    
    # good_entropy_question_indices = [e['question_index'] for e in questions_evalutaion if e['entropy'] > 0.3 and e['entropy'] < 0.7]
    # bad_entropy_questions = [q for i,q in enumerate(existing_questions) if i not in good_entropy_question_indices]
    # good_entropy_questions = [q for i,q in enumerate(existing_questions) if i in good_entropy_question_indices]


    # relevance_info, keep_indices, _ = evaluate_question_relevance(torch.tensor(answers_arr.T), torch.tensor(pseudo_labels))
    # print(relevance_info)

    # to_keep = good_entropy_question_indices
    # existing_questions = [q for i,q in enumerate(existing_questions) if i in to_keep]

    # existing_questions = good_entropy_questions

    # prompt = (
    #     f"You must analyze supporting questions for a main classification task. Here is the task description:\n"
    #     f"```\n"
    #     f"{class_task_desc}"
    #     f"```\n\n"
    #     f"Here is a set of questions so far:\n"
    #     f"```\n"
    #     f"{'\n'.join(existing_questions)}\n"
    #     f"Now give suggestions on what new questions could explore that hasn't been explored so far."
    # )

    # print(prompt)

    # suggestions = llm.invoke(prompt, temperature=0.0)

    # print(suggestions)

    # print('Kept:')
    # for q in existing_questions:
    #     print('\t', q)
    # print()

INFO 05-02 11:01:17 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'model': 'AxionML/Qwen3.5-9B-NVFP4'}
INFO 05-02 11:01:18 [model.py:549] Resolved architecture: Qwen3_5ForConditionalGeneration
INFO 05-02 11:01:18 [model.py:1678] Using max model len 4096
INFO 05-02 11:01:18 [config.py:281] Setting attention block size to 528 tokens to ensure that attention page size is >= mamba page size.
INFO 05-02 11:01:18 [config.py:312] Padding mamba page size by 0.76% to ensure that mamba page size and attention page size are exactly equal.
WARNING 05-02 11:01:18 [modelopt.py:998] Detected ModelOpt NVFP4 checkpoint. Please note that the format is experimental and could change in future.
(EngineCore pid=4002975) INFO 05-02 11:01:31 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='AxionML/Qwen3.5-9B-NVFP4', speculative_config=None, tokenizer='AxionML/Qwen3.5-9B-NVFP4', skip_tokenize

(EngineCore pid=4002975) [transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.


(EngineCore pid=4002975) INFO 05-02 11:01:34 [parallel_state.py:1400] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://10.10.10.23:38861 backend=nccl
(EngineCore pid=4002975) INFO 05-02 11:01:34 [parallel_state.py:1716] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A


(EngineCore pid=4002975) [transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


(EngineCore pid=4002975) INFO 05-02 11:01:44 [gpu_model_runner.py:4735] Starting to load model AxionML/Qwen3.5-9B-NVFP4...
(EngineCore pid=4002975) INFO 05-02 11:01:44 [cuda.py:390] Using backend AttentionBackendEnum.FLASH_ATTN for vit attention
(EngineCore pid=4002975) INFO 05-02 11:01:44 [mm_encoder_attention.py:230] Using AttentionBackendEnum.FLASH_ATTN for MMEncoderAttention.
(EngineCore pid=4002975) INFO 05-02 11:01:44 [nvfp4_utils.py:85] Using NvFp4LinearBackend.MARLIN for NVFP4 GEMM
(EngineCore pid=4002975) INFO 05-02 11:01:44 [gdn_linear_attn.py:147] Using Triton/FLA GDN prefill kernel
(EngineCore pid=4002975) INFO 05-02 11:01:44 [cuda.py:334] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
(EngineCore pid=4002975) INFO 05-02 11:01:44 [flash_attn.py:596] Using FlashAttention version 2


...

(EngineCore pid=4004343) INFO 05-02 11:04:16 [gpu_model_runner.py:6046] Graph capturing finished in 3 secs, took 0.52 GiB
(EngineCore pid=4004343) INFO 05-02 11:04:16 [gpu_worker.py:597] CUDA graph pool memory: 0.52 GiB (actual), 0.71 GiB (estimated), difference: 0.2 GiB (38.6%).
(EngineCore pid=4004343) INFO 05-02 11:04:16 [core.py:283] init engine (profile, create kv cache, warmup model) took 10.95 seconds
(EngineCore pid=4004343) INFO 05-02 11:04:16 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=4004343) INFO 05-02 11:04:16 [compilation.py:292] Enabled custom fusions: act_quant


Processed prompts: 100%|██████████| 23/23 [00:16<00:00,  1.41it/s, est. speed input: 548.23 toks/s, output: 910.54 toks/s] 


Using:
	*  Does the described pathogen possess its own cellular machinery, such as ribosomes and cytoplasm, to replicate independently?
	*  Is the pathogen described as a unicellular prokaryotic organism lacking a membrane-bound nucleus?
	*  Does the text explicitly mention the presence of a peptidoglycan cell wall or the results of a Gram stain (Gram-positive or Gram-negative)?
	*  Is the primary etiology causing the condition explicitly identified as a viral agent?
	*  Does the text rule out bacterial or parasitic causes in favor of a viral infection?
	*  Is the condition described as a specific viral infection of a defined organ system rather than a general symptom or non-infectious pathology?
	*  Is the causative agent of the condition explicitly identified as a parasite (such as protozoa, helminths, or ectoparasites), excluding viruses, bacteria, fungi, or neoplastic growths?
	*  Does the clinical description define a state of active infection or infestation by the organism, rathe

Rendering conversations:   0%|          | 0/100 [00:00<?, ?it/s]

VLLMValidationError: This model's maximum context length is 4096 tokens. However, you requested 0 output tokens and your prompt contains at least 4097 input tokens, for a total of at least 4097 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=4097)

In [23]:
import torch
import networkx as nx
import plotly.graph_objects as go


def plot_hetero_graph_plotly(data, questions, predicted_y):
    """
    data: PyG HeteroData
    questions: list of question strings
    predicted_y: tensor of predicted labels for text nodes
    """

    G = nx.DiGraph()  # directed now (important)

    num_text = data['text'].x.size(0)
    num_question = data['question'].x.size(0)
    question_offset = num_text

    id2label = data.id2label
    id2label[-1] = 'Unlabeled'

    # --------------------
    # Add TEXT nodes
    # --------------------
    for i in range(num_text):
        pseudo = int(data['text'].pseudo_y[i])
        true = int(data['text'].y[i])
        pred = int(predicted_y[i])
        correct = (true == pred)

        G.add_node(
            i,
            node_type="text",
            pseudo=pseudo,
            true=true,
            pred=pred,
            correct=correct
        )

    # --------------------
    # Add QUESTION nodes
    # --------------------
    for i in range(num_question):
        G.add_node(
            question_offset + i,
            node_type="question",
            question=questions[i]
        )

    # --------------------
    # Add edges WITH relation type
    # --------------------
    # text -> text
    edge_index = data['text', 'similar', 'text'].edge_index
    for src, dst in edge_index.t().tolist():
        G.add_edge(src, dst, relation="similar")

    # question -> text
    edge_index = data['question', 'answers_yes', 'text'].edge_index
    for src, dst in edge_index.t().tolist():
        G.add_edge(question_offset + src, dst, relation="answers_yes")

    # text -> question
    # edge_index = data['text', 'rev_answers_yes', 'question'].edge_index
    # for src, dst in edge_index.t().tolist():
    #     G.add_edge(src, question_offset + dst, relation="rev_answers_yes")

    # --------------------
    # Layout
    # --------------------
    pos = nx.spring_layout(G, k=0.3, seed=42)

    # --------------------
    # Separate node traces
    # --------------------
    text_x, text_y, text_color, text_hover = [], [], [], []
    question_x, question_y, question_hover = [], [], []

    for node, attr in G.nodes(data=True):
        x, y = pos[node]

        if attr["node_type"] == "text":
            text_x.append(x)
            text_y.append(y)
            text_color.append("blue" if attr["correct"] else "red")

            hover_text = (
                f"<b>Text Node {node}</b><br>"
                f"Pseudo: {id2label[attr['pseudo']]}<br>"
                f"Predicted: {id2label[attr['pred']]}<br>"
                f"True: {id2label[attr['true']]}"
            )
            text_hover.append(hover_text)
        else:
            question_x.append(x)
            question_y.append(y)
            question_hover.append(
                f"<b>Question</b><br>{attr['question']}"
            )

    # --------------------
    # Build edge traces by relation
    # --------------------
    edge_styles = {
        "similar": dict(color="gray", width=0.5, dash="solid"),
        "answers_yes": dict(color="gray", width=2, dash="dash"),
        "rev_answers_yes": dict(color="gray", width=2, dash="dot"),
    }

    edge_traces = []

    for relation in edge_styles:
        if relation == 'rev_answers_yes':
            continue

        edge_x, edge_y = [], []

        for src, dst, attr in G.edges(data=True):
            if attr["relation"] == relation:
                x0, y0 = pos[src]
                x1, y1 = pos[dst]
                edge_x += [x0, x1, None]
                edge_y += [y0, y1, None]

        style = edge_styles[relation]

        edge_traces.append(
            go.Scatter(
                x=edge_x,
                y=edge_y,
                mode="lines",
                line=dict(
                    width=style["width"],
                    color=style["color"],
                    dash=style["dash"]
                ),
                hoverinfo="none",
                name=relation
            )
        )

    # --------------------
    # Node traces
    # --------------------
    text_trace = go.Scatter(
        x=text_x,
        y=text_y,
        mode='markers',
        hoverinfo='text',
        text=text_hover,
        marker=dict(size=10, color=text_color, symbol='circle'),
        name='Text Nodes',
        opacity=0.5
    )

    question_trace = go.Scatter(
        x=question_x,
        y=question_y,
        mode='markers',
        hoverinfo='text',
        text=question_hover,
        marker=dict(size=14, color='yellow', symbol='square'),
        name='Question Nodes'
    )

    # --------------------
    # Final figure
    # --------------------
    fig = go.Figure(
        data=edge_traces + [text_trace, question_trace],
        layout=go.Layout(
            title="Heterogeneous Graph Visualization",
            showlegend=True,
            hovermode='closest',
            margin=dict(b=20, l=5, r=5, t=40),
            xaxis=dict(showgrid=False, zeroline=False, visible=False),
            yaxis=dict(showgrid=False, zeroline=False, visible=False),
            height=850
        )
    )

    fig.show()

In [25]:
questions = [
    'Is the primary pathology located in the respiratory system, including the lungs, airways, or upper respiratory tract?',
	'Is the primary pathology located in the urinary system or male genital organs?',
	'Is the primary condition driven by nutritional deficiencies, metabolic disorders, or environmental exposures rather than structural organ disease?',
	'Is the primary condition characterized by immune system dysfunction, hypersensitivity, or autoimmune mechanisms?',
	'Does the article discuss diseases specific to animals (veterinary medicine) rather than human medicine?',
	'Target Respiratory Tract Diseases (Category 7).',
	'Target Urologic and Male Genital Diseases (Category 11).',
	'Target Nutritional and Metabolic Diseases (Category 17).',
	'Target Immunologic Diseases (Category 19).',
	'Target Disorders of Environmental Origin (Category 20).',
	'Is the primary pathology located in the respiratory system, including the lungs, airways, or respiratory tract?',
	'Is the primary pathology located in the urinary system or male genital organs?',
	'Is the primary condition driven by nutritional deficiencies, metabolic disorders, or environmental exposures rather than structural organ disease?',
	'Is the primary condition characterized by immune system dysfunction, hypersensitivity, or autoimmune mechanisms?',
	'Does the article discuss diseases specific to animals (veterinary medicine) rather than human medicine?',
	'Is the primary etiology of the condition an infectious agent (bacteria, virus, fungus, or parasite) or a non-infectious factor (such as genetic, environmental, or neoplastic)?',
	'If the condition is infectious, does the causative agent belong to the bacterial/fungal domain or the viral/parasitic domain?',
	'If the condition is non-infectious, is the primary pathology localized to the nervous system, cardiovascular system, or integumentary system?',
	'If the condition is non-infectious and not confined to the nervous, cardiovascular, or integumentary systems, does it involve the reproductive system, digestive system, or hematologic/lymphatic system?',
	'If the condition does not fit specific organ system categories above, is it primarily defined by a specific patient demographic (such as neonates) or a general pathological sign/symptom rather than a specific disease entity?',
	'Does the article explicitly describe a neoplasm, tumor, or cancer as the primary condition?',
	'Is the primary pathology located in the endocrine glands or hormonal regulatory systems?',
	'Is the primary pathology located in the musculoskeletal system, including bones, joints, or muscles?',
	'Is the primary pathology located in the ocular structures (eyes)?',
	'Is the condition specifically defined as a disease affecting animals rather than humans?',
]

plot_hetero_graph_plotly(hetero_data, questions, propagated_y)